# Notebook 3: Compressed sensing

*Part of MRI Reconstruction from k-space to Deep Learning: A Hands-on Introduction*

Notebook 2 ended with a promise. Random sampling turns aliasing into noise-like artifacts, and real images have structure that noise does not. So an algorithm that knows what images look like should be able to remove the artifacts. Compressed sensing (Lustig, Donoho & Pauly, 2007) turns this idea into a reconstruction method. It has been available on commercial MRI scanners since 2017, usually combined with multi-coil parallel imaging.

**Learning objectives.** By the end of this notebook you will be able to:

1. Explain what it means for an image to be sparse or compressible in a transform, using the Haar wavelet.
2. Write down the compressed sensing optimization problem and explain the role of each term.
3. Implement ISTA, an iterative algorithm that alternates between data consistency and soft-thresholding.
4. Explain the effect of the regularization weight λ, including how too much regularization can change the anatomy.
5. Explain why compressed sensing needs incoherent sampling, and where its limits are.

**Prerequisites:** Notebooks 1 and 2.

**Time:** about 1.5–2.5 hours on your own.

As always, answer every Predict question before running the cell.

## 1. Setup

The second cell repeats the helper functions from Notebooks 1 and 2: the FFT helpers, plotting, NRMSE, sampling patterns and zero-filled reconstruction.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from skimage.data import shepp_logan_phantom
from skimage.transform import resize

plt.rcParams["figure.dpi"] = 100
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]  # line colors, always used in this order

In [ ]:
# From Notebooks 1 and 2
def fft2c(image):
    """Centered, orthonormal 2D FFT: image -> k-space."""
    return np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(image), norm="ortho"))


def ifft2c(kspace):
    """Centered, orthonormal inverse 2D FFT: k-space -> image."""
    return np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(kspace), norm="ortho"))


def show(images, titles, cmap="gray", vmin=None, vmax=None):
    """Show images side by side."""
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=(3.4 * n, 3.6))
    for ax, im, title in zip(np.atleast_1d(axes), images, titles):
        ax.set_title(title)
        if im.dtype == bool:  # sampling masks: white = measured, black = skipped, with a thin frame
            ax.imshow(im, cmap=cmap, vmin=0, vmax=1, interpolation="nearest")
            ax.set_xticks([])
            ax.set_yticks([])
        else:
            ax.imshow(im, cmap=cmap, vmin=vmin, vmax=vmax)
            ax.axis("off")
    plt.tight_layout()
    plt.show()


def nrmse(recon, reference):
    """Normalized root-mean-square error: 0 is perfect, 1 is as large as the image itself."""
    return np.linalg.norm(recon - reference) / np.linalg.norm(reference)


def tidy(ax):
    """Quiet grid and axes for line plots."""
    ax.grid(alpha=0.25, linewidth=0.6)
    ax.spines[["top", "right"]].set_visible(False)


def regular_lines(N, R):
    """Every R-th phase-encode line, always including the center line."""
    lines = np.zeros(N, dtype=bool)
    lines[(N // 2) % R :: R] = True
    return lines


def variable_density_lines(N, R, center_fraction=0.08, decay=3, seed=0):
    """N // R lines: a fully sampled center, the rest random and more likely near the center."""
    rng = np.random.default_rng(seed)
    n_total = N // R
    n_center = int(round(N * center_fraction))
    lines = np.zeros(N, dtype=bool)
    lines[N // 2 - n_center // 2 : N // 2 - n_center // 2 + n_center] = True
    candidates = np.flatnonzero(~lines)
    distance = np.abs(candidates - N // 2) / (N / 2)
    probability = (1 - distance) ** decay + 1e-6
    lines[rng.choice(candidates, size=n_total - n_center, replace=False,
                     p=probability / probability.sum())] = True
    return lines


def as_mask(lines):
    """Turn a 1D set of phase-encode lines into a 2D k-space mask."""
    return np.repeat(lines[:, None], lines.size, axis=1)


def zero_filled(kspace, lines):
    """Treat unmeasured k-space as zero and apply the inverse FFT."""
    return np.abs(ifft2c(kspace * lines[:, None]))


# The object and its fully sampled k-space
N = 256
image = resize(shepp_logan_phantom(), (N, N), anti_aliasing=True)
kspace = fft2c(image)
print(f"Ground truth: {N} x {N} phantom")

## 2. Sparsity: most numbers can be zero

An image is sparse if most of its numbers are zero, and compressible if most are *almost* zero. Our phantom is not sparse as an array of pixels: most pixels inside the head are bright. But it may be sparse in another representation.

A simple and useful one is the Haar wavelet transform. It replaces each pair of neighboring pixels by their average and their difference, first along rows, then along columns. Then it repeats the same on the smaller image of averages, again and again. In a region of constant brightness, all the differences are zero. Only edges produce large differences.

The two functions below implement the 2D Haar transform and its exact inverse in a few lines of NumPy.

In [ ]:
SQRT2 = np.sqrt(2)


def haar2(x, levels=4):
    """2D Haar wavelet transform: averages and differences of neighboring pixels, repeated `levels` times."""
    c = x.astype(complex).copy()
    n = c.shape[0]
    for _ in range(levels):
        a = c[:n, :n]
        a = np.hstack([(a[:, 0::2] + a[:, 1::2]) / SQRT2, (a[:, 0::2] - a[:, 1::2]) / SQRT2])  # along rows
        a = np.vstack([(a[0::2, :] + a[1::2, :]) / SQRT2, (a[0::2, :] - a[1::2, :]) / SQRT2])  # along columns
        c[:n, :n] = a
        n //= 2  # the next level works on the small "averages" image in the top-left corner
    return c


def ihaar2(c, levels=4):
    """Inverse of haar2: rebuilds the image exactly from its wavelet coefficients."""
    x = c.copy()
    n = x.shape[0] // 2 ** (levels - 1)
    for _ in range(levels):
        a = x[:n, :n]
        h = n // 2
        b = np.empty_like(a)
        b[0::2, :] = (a[:h, :] + a[h:, :]) / SQRT2
        b[1::2, :] = (a[:h, :] - a[h:, :]) / SQRT2
        a2 = np.empty_like(a)
        a2[:, 0::2] = (b[:, :h] + b[:, h:]) / SQRT2
        a2[:, 1::2] = (b[:, :h] - b[:, h:]) / SQRT2
        x[:n, :n] = a2
        n *= 2
    return x

In [ ]:
coeffs = haar2(image, levels=4)

print(f"Inverse transform error: {np.abs(ihaar2(coeffs, levels=4) - image).max():.1e}  (exact)")
print(f"Energy kept by the transform: {np.sum(np.abs(coeffs)**2) / np.sum(image**2):.6f}  (orthonormal, like the FFT)")
show([image, np.log10(np.abs(coeffs) + 1e-4)],
     ["phantom", "Haar coefficients (log scale)"])

The coefficient image has a small copy of the phantom in the top-left corner: the averages after four levels. Everything else is differences at different scales and directions. They are bright only along edges and dark (near zero) almost everywhere else.

> **Predict:** Keep only the largest 5% of the numbers and set the other 95% to zero. Which gives a better image: keeping 5% of the pixels, or 5% of the Haar coefficients?

In [ ]:
def keep_largest(values, fraction):
    """Keep the largest `fraction` of values (by magnitude), set all others to zero."""
    k = int(fraction * values.size)
    cutoff = np.sort(np.abs(values).ravel())[-k]
    return np.where(np.abs(values) >= cutoff, values, 0)


print(f"{'kept':>6}{'pixels':>10}{'Haar':>10}   (NRMSE)")
for fraction in [0.02, 0.05, 0.10]:
    from_pixels = np.abs(keep_largest(image, fraction))
    from_haar = np.abs(ihaar2(keep_largest(coeffs, fraction)))
    print(f"{fraction:>6.0%}{nrmse(from_pixels, image):>10.3f}{nrmse(from_haar, image):>10.3f}")

show([np.abs(keep_largest(image, 0.05)), np.abs(ihaar2(keep_largest(coeffs, 0.05)))],
     ["largest 5% of pixels", "largest 5% of Haar coefficients"], vmin=0, vmax=1)

With 5% of the pixels, most of the image is simply missing. With 5% of the Haar coefficients, the phantom is almost perfect: the phantom is compressible in the Haar domain. This is the same reason JPEG files are small: natural images are compressible in a wavelet-like transform.

A fair warning: the Shepp-Logan phantom is made of flat ellipses, which is the best possible case for Haar wavelets. Real MR images have texture and noise and are less compressible. We will come back to this in Section 9.

## 3. The compressed sensing recipe

Compressed sensing combines three ingredients, all of which you have now seen:

1. **Incoherent sampling** (Notebook 2): random, variable-density undersampling makes artifacts noise-like.
2. **A sparse representation** (Section 2): the true image has few large wavelet coefficients, while noise-like artifacts spread over many small ones.
3. **A nonlinear reconstruction** that looks for the image that is *both* consistent with the measurements *and* sparse:

$$ \hat{x} = \arg\min_x \; \underbrace{\tfrac{1}{2}\,\| M F x - y \|_2^2}_{\text{data consistency}} \; + \; \lambda \, \underbrace{\| \Psi x \|_1}_{\text{sparsity}} $$

- $x$ is the image we are looking for, $F$ the Fourier transform, $M$ the sampling mask and $y$ the measured k-space lines.
- The first term is small when the image's k-space agrees with what was actually measured.
- The second term is the sum of the magnitudes of the wavelet coefficients $\Psi x$ (the ℓ1 norm). It is small when the image is sparse.
- **λ** sets the balance between the two.

Why the ℓ1 norm and not simply "the number of non-zero coefficients"? Counting non-zeros leads to a problem that is impossible to solve efficiently. The ℓ1 norm is the closest version that can be solved efficiently, and it still pushes small coefficients to exactly zero, as the next section shows.

## 4. Soft thresholding

The operation that makes coefficients sparse is soft thresholding: shrink every value towards zero by λ, and set everything smaller than λ exactly to zero. It is what the ℓ1 term does to a single coefficient.

> **Predict:** Before running the cell, sketch the output of soft thresholding with λ = 1 for inputs from −3 to 3.

In [ ]:
def soft_threshold(c, lam):
    """Shrink every value towards zero by lam; values smaller than lam become exactly zero.
    Works for complex numbers: the magnitude shrinks, the phase stays."""
    magnitude = np.abs(c)
    return c * np.maximum(magnitude - lam, 0) / np.maximum(magnitude, 1e-12)

In [ ]:
values = np.linspace(-3, 3, 601)

fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(values, values, color=COLORS[0], linewidth=2, linestyle="--", label="input (no change)")
ax.plot(values, soft_threshold(values, 1.0).real, color=COLORS[1], linewidth=2, label="soft threshold, λ = 1")
ax.set_xlabel("input value")
ax.set_ylabel("output value")
ax.set_title("Small values become zero, large values shrink by λ", loc="left")
ax.legend(frameon=False)
tidy(ax)
plt.tight_layout()
plt.show()

Values between −λ and λ become exactly zero. Larger values survive but shrink by λ. Applied to wavelet coefficients, this removes the many small, noise-like coefficients and keeps the few large ones that describe real edges.

## 5. ISTA: alternating between two goals

The Iterative Shrinkage-Thresholding Algorithm (ISTA) solves the problem from Section 3 by repeating two simple steps:

1. **Data consistency.** Go to k-space and put the measured lines back exactly. Everything we measured is kept; only the missing lines are filled with the current guess.
2. **Sparsity.** Go to the wavelet domain and soft-threshold the coefficients. This removes some of the noise-like artifact.

Each step undoes a little of the other, but over many iterations the image converges to one that is both consistent and sparse. (Step 1 is exactly a gradient step on the data consistency term with step size 1, which works because our FFT is orthonormal.)

The whole algorithm fits in about fifteen lines. Read it carefully: this structure, a data consistency step plus a "make it look like a real image" step, is also the backbone of the deep learning methods in Notebook 4.

In [ ]:
def cs_reconstruct(kspace_measured, lines, lam, n_iter=200, levels=4, reference=None):
    """Compressed sensing reconstruction with ISTA (iterative soft-thresholding).

    kspace_measured : k-space with unmeasured lines set to zero
    lines           : boolean array, True for measured phase-encode lines
    lam             : regularization weight (how strongly to enforce sparsity)
    reference       : optional ground truth, only used to record the error at each iteration
    """
    n = kspace_measured.shape[0]
    coarse = np.zeros((n, n), dtype=bool)          # the small "averages" image is not thresholded
    coarse[: n // 2**levels, : n // 2**levels] = True

    x = ifft2c(kspace_measured)                    # start from the zero-filled image
    history = []
    for _ in range(n_iter):
        # Step 1, data consistency: put the measured k-space lines back exactly
        k = fft2c(x)
        k = np.where(lines[:, None], kspace_measured, k)
        x = ifft2c(k)
        # Step 2, sparsity: soft-threshold the wavelet coefficients
        w = haar2(x, levels)
        w = np.where(coarse, w, soft_threshold(w, lam))
        x = ihaar2(w, levels)
        if reference is not None:
            history.append(nrmse(np.abs(x), reference))
    return x, history

## 6. Compressed sensing at R = 4

We use the same variable-density mask as Notebook 2 (64 of 256 lines) and λ = 0.01.

> **Predict:** Compared with zero-filling (NRMSE 0.293 in Notebook 2), how much lower do you expect the error to be? Will the three small ellipses near the bottom be visible?

In [ ]:
lines = variable_density_lines(N, R=4)
measured = kspace * lines[:, None]

start = time.time()
recon, history = cs_reconstruct(measured, lines, lam=0.01, reference=image)
recon = np.abs(recon)
print(f"200 iterations took {time.time() - start:.1f} s")

zf = zero_filled(kspace, lines)
print(f"NRMSE zero-filled:          {nrmse(zf, image):.3f}")
print(f"NRMSE compressed sensing:   {nrmse(recon, image):.3f}")

show([image, zf, recon], ["ground truth", "zero-filled", "compressed sensing"], vmin=0, vmax=1)
show([np.abs(zf - image), np.abs(recon - image)], ["error: zero-filled", "error: compressed sensing"], vmin=0, vmax=0.3)

Zoom in on the three small ellipses near the bottom of the phantom:

In [ ]:
def zoom(img):
    """The region with the three small ellipses near the bottom of the phantom."""
    return img[170:230, 90:170]


show([zoom(image), zoom(zf), zoom(recon)], ["ground truth", "zero-filled", "compressed sensing"], vmin=0, vmax=0.5)

Compressed sensing removes the streaks and roughly halves the error, from the *same* 64 measured lines. The small ellipses, buried in streaks after zero-filling, are visible again. The remaining error sits mostly along edges.

Watch how the error falls with each iteration:

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.8))
ax.axhline(nrmse(zf, image), color=COLORS[1], linewidth=2, linestyle="--", label="zero-filled (starting point)")
ax.plot(np.arange(1, len(history) + 1), history, color=COLORS[0], linewidth=2, label="compressed sensing")
ax.set_xlabel("iteration")
ax.set_ylabel("NRMSE (lower is better)")
ax.set_title("Most of the gain comes in the first 100 iterations", loc="left")
ax.set_ylim(bottom=0)
ax.legend(frameon=False)
tidy(ax)
plt.tight_layout()
plt.show()

## 7. Choosing λ: too little, too much

> **Predict:** What will the image look like with a very small λ (0.001)? With a very large one (0.1)?

In [ ]:
lams = [0.001, 0.01, 0.1]
results = {lam: np.abs(cs_reconstruct(measured, lines, lam)[0]) for lam in lams}

show([results[lam] for lam in lams], [f"λ = {lam}" for lam in lams], vmin=0, vmax=1)
show([zoom(image)] + [zoom(results[lam]) for lam in lams],
     ["ground truth"] + [f"λ = {lam}" for lam in lams], vmin=0, vmax=0.5)

- **λ too small (0.001):** sparsity is barely enforced, so much of the streaky artifact remains.
- **λ = 0.01:** a good balance.
- **λ too large (0.1):** the image is forced to be *too* sparse. It becomes blocky (Haar wavelets are square), and in the zoom the three small ellipses merge into one blob. The reconstruction did not just blur the image; it changed the anatomy. A real lesion could disappear in the same way.

Let's measure the whole range:

In [ ]:
lam_values = [0.001, 0.002, 0.005, 0.01, 0.02, 0.03, 0.05, 0.1]
lam_errors = [nrmse(np.abs(cs_reconstruct(measured, lines, lam)[0]), image) for lam in lam_values]

for lam, e in zip(lam_values, lam_errors):
    print(f"λ = {lam:<6}: NRMSE = {e:.3f}")

fig, ax = plt.subplots(figsize=(6, 3.8))
ax.semilogx(lam_values, lam_errors, "o-", color=COLORS[0], linewidth=2, markersize=8)
ax.set_xlabel("λ (log scale)")
ax.set_ylabel("NRMSE (lower is better)")
ax.set_title("Too little and too much regularization both hurt", loc="left")
ax.set_ylim(bottom=0)
tidy(ax)
plt.tight_layout()
plt.show()

The best λ sits in the middle. But notice what we did: we chose λ by comparing with the ground truth, which never exists for a real patient. In practice, λ is tuned on calibration scans or training data and then fixed. Tuning parameters like this is one of the practical burdens of compressed sensing.

## 8. Sampling still matters

> **Predict:** What happens if we run exactly the same algorithm on the regular R = 4 mask from Notebook 2, which produces ghost copies?

In [ ]:
regular = regular_lines(N, R=4)
regular_measured = kspace * regular[:, None]
regular_cs = np.abs(cs_reconstruct(regular_measured, regular, lam=0.01)[0])

print(f"Regular mask, zero-filled:        NRMSE = {nrmse(zero_filled(kspace, regular), image):.3f}")
print(f"Regular mask, compressed sensing: NRMSE = {nrmse(regular_cs, image):.3f}")
show([zero_filled(kspace, regular), regular_cs],
     ["regular, zero-filled", "regular, compressed sensing"], vmin=0, vmax=1)

Nothing improves. Each ghost copy is a perfectly sparse, perfectly plausible image, and the ghosts are consistent with the measured lines. Compressed sensing has no way to know which copy is real. Sparsity only helps when the artifacts are incoherent, which is why ingredient 1 of the recipe is not optional.

## 9. How far can compressed sensing go?

> **Predict:** Notebook 2 showed zero-filled error growing with R. Will compressed sensing keep its large advantage at R = 6 and R = 8?

In [ ]:
Rs = [4, 6, 8]
zf_errors, cs_errors, cs_images = [], [], []
for R in Rs:
    l = variable_density_lines(N, R)
    m = kspace * l[:, None]
    # try a few λ and keep the best (an optimistic choice: it uses the ground truth)
    candidates = [np.abs(cs_reconstruct(m, l, lam)[0]) for lam in [0.005, 0.01, 0.02, 0.03]]
    best = min(candidates, key=lambda x: nrmse(x, image))
    zf_errors.append(nrmse(zero_filled(kspace, l), image))
    cs_errors.append(nrmse(best, image))
    cs_images.append(best)
    print(f"R = {R}: zero-filled {zf_errors[-1]:.3f}, compressed sensing {cs_errors[-1]:.3f}")

fig, ax = plt.subplots(figsize=(5.5, 3.8))
ax.plot(Rs, zf_errors, "o-", color=COLORS[1], linewidth=2, markersize=8, label="zero-filled")
ax.plot(Rs, cs_errors, "o-", color=COLORS[0], linewidth=2, markersize=8, label="compressed sensing")
ax.set_xlabel("acceleration factor R")
ax.set_ylabel("NRMSE (lower is better)")
ax.set_title("The advantage shrinks at high acceleration", loc="left")
ax.set_xticks(Rs)
ax.set_ylim(bottom=0)
ax.legend(frameon=False)
tidy(ax)
plt.tight_layout()
plt.show()

show(cs_images, [f"compressed sensing, R = {R}" for R in Rs], vmin=0, vmax=1)

Compressed sensing helps at every R, but the gain shrinks quickly: at R = 8, with only 32 lines, the result shows blocky patches and bands. With a single coil and only 1D undersampling, there is simply not enough information left.

## 10. Limits of compressed sensing, and the step to deep learning

Compressed sensing works, and it runs on clinical scanners today. But this notebook has also shown its practical limits:

1. **λ must be tuned**, and the wrong value can remove or merge small structures (Section 7).
2. **It is slow.** It needs hundreds of iterations, each with several FFTs and wavelet transforms. On real 3D, multi-coil data, that can mean minutes per scan.
3. **The idea of a "real image" is hand-chosen.** Haar wavelets are perfect for flat ellipses but produce blocky artifacts on real anatomy. Better transforms exist (smoother wavelets, total variation), but each is still a human guess about what images look like.
4. **Gains shrink at high acceleration** (Section 9).

**Deep learning reconstruction** keeps the same structure, data consistency plus a step that makes the image look real, but learns that second step from thousands of example images instead of using a fixed rule like soft-thresholding. After training, it runs in a fraction of a second. That is Notebook 4.

## Exercises

Try each exercise before opening the hint. Solutions are in `solutions/03_solutions.ipynb`.

### Exercise 1: another kind of sparsity

Compute the vertical and horizontal differences between neighboring pixels of the phantom (`np.diff(image, axis=0)` and `np.diff(image, axis=1)`). What fraction of the differences is close to zero (magnitude below 0.001)? Compare with the fraction of Haar detail coefficients below 0.001. This "sparsity of differences" is the basis of another popular regularizer, total variation (TV).

<details><summary>Hint</summary>

<code>np.mean(np.abs(d) < 1e-3)</code> gives the fraction of values below the threshold.
</details>

In [ ]:
# Exercise 1: your code here


### Exercise 2: is the result consistent with the measurements?

ISTA ends with a soft-thresholding step, so its final image may not agree *exactly* with the measured k-space lines. Compute the relative difference between the measured lines and the same lines in the k-space of the R = 4 result. Then apply one final data-consistency step (put the measured lines back) and compare the NRMSE and the image. Why might a radiologist prefer the version with exact data consistency, even if its NRMSE were slightly worse?

<details><summary>Hint</summary>

Take <code>fft2c</code> of the complex result (before <code>np.abs</code>) and compare only the rows where <code>lines</code> is True.
</details>

In [ ]:
# Exercise 2: your code here


### Exercise 3: the best mask depends on the reconstruction

At R = 4, compare `variable_density_lines(N, 4, decay=3)` (the default) with `variable_density_lines(N, 4, decay=1)`, which spreads more lines into the outer k-space. Compute the NRMSE of both masks with zero-filling and with compressed sensing (λ = 0.01). Which mask wins for each method? Why?

<details><summary>Hint</summary>

Think back to Notebook 2: which kind of artifact does a mask with more random outer lines create, and which method can remove that kind of artifact?
</details>

In [ ]:
# Exercise 3: your code here


### Exercise 4: noisy measurements

Real k-space contains noise. Add complex Gaussian noise to the k-space before undersampling:

```python
noise_rng = np.random.default_rng(0)
sigma = 0.05
noisy_kspace = kspace + sigma * (noise_rng.standard_normal((N, N)) + 1j * noise_rng.standard_normal((N, N))) / np.sqrt(2)
```

Repeat the λ sweep of Section 7 on the noisy data (compare with the clean `image`). Does the best λ change? Why?

In [ ]:
# Exercise 4: your code here


## Summary

- An image is compressible if a few transform coefficients describe it well. The phantom is nearly perfect with only 5% of its Haar wavelet coefficients.
- **Compressed sensing** looks for an image that is both consistent with the measured k-space and sparse, balanced by λ.
- **ISTA** alternates a data consistency step (put measured lines back) and a sparsity step (soft-threshold the wavelet coefficients).
- At R = 4 it roughly halves the error of zero-filling and recovers small structures, but only with incoherent sampling.
- **λ** must be tuned: too small leaves artifacts, too large makes images blocky and can merge or erase small structures.
- Its limits (tuning, speed, a hand-chosen idea of "real images", reduced gains at high R) motivate deep learning reconstruction.

**Next:** Notebook 4, *Deep learning reconstruction*.

## References

1. Lustig, M., Donoho, D., & Pauly, J. M. (2007). Sparse MRI: The application of compressed sensing for rapid MR imaging. *Magnetic Resonance in Medicine*, 58(6), 1182–1195.
2. Daubechies, I., Defrise, M., & De Mol, C. (2004). An iterative thresholding algorithm for linear inverse problems with a sparsity constraint. *Communications on Pure and Applied Mathematics*, 57(11), 1413–1457.
3. Candès, E. J., Romberg, J., & Tao, T. (2006). Robust uncertainty principles: Exact signal reconstruction from highly incomplete frequency information. *IEEE Transactions on Information Theory*, 52(2), 489–509.
4. Donoho, D. L. (2006). Compressed sensing. *IEEE Transactions on Information Theory*, 52(4), 1289–1306.

---
*Text and figures: CC-BY 4.0. Code: MIT License.*